In [1]:
import pandas as pd
import requests
from string import ascii_uppercase as alphabet
import pickle

url = 'https://en.wikipedia.org/wiki/2025_FIFA_U-20_World_Cup#Group_stage'
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
response.raise_for_status() 

all_tables = pd.read_html(response.text)
# ...existing code...

C:\Users\mathe\AppData\Local\Temp\ipykernel_16796\2315214905.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(response.text)


In [2]:
all_tables[5]
all_tables[12]

all_tables[40]



,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts,Qualification
0,1,Colombia,0,0,0,0,0,0,0,0,Knockout stage
1,2,Saudi Arabia,0,0,0,0,0,0,0,0,Knockout stage
2,3,Norway,0,0,0,0,0,0,0,0,Possible knockout stage
3,4,Nigeria,0,0,0,0,0,0,0,0,NaN


In [8]:
alphabet

'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

In [9]:
all_tables = pd.read_html(response.text)

dict_table = {}
for letter, i in zip (alphabet,range (5, 47, 7)):
    df = all_tables[i]
    df.pop('Qualification')
    dict_table[f'Group {letter}'] = df

C:\Users\mathe\AppData\Local\Temp\ipykernel_15116\1531442705.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(response.text)


In [10]:
dict_table['Group F']

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts
0,1,Colombia,0,0,0,0,0,0,0,0
1,2,Saudi Arabia,0,0,0,0,0,0,0,0
2,3,Norway,0,0,0,0,0,0,0,0
3,4,Nigeria,0,0,0,0,0,0,0,0


In [11]:
with open('dict_table', 'wb') as output:
    pickle.dump(dict_table, output)

In [12]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [13]:
years = [1977, 1979, 1981, 1983, 1985, 1987, 1989, 1991, 1993,
        1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 
        2013, 2015, 2017, 2019, 2023] 

def get_matches(year):
    if year < 2007:
        url = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Youth_Championship'
    else:
        url = f'https://en.wikipedia.org/wiki/{year}_FIFA_U-20_World_Cup'
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    content = response.text

    from bs4 import BeautifulSoup
    soup = BeautifulSoup(content, 'lxml')

    matches = soup.find_all('div', class_='footballbox')

    home = []
    away = []
    score = []

    for match in matches:
        home.append(match.find('th', class_='fhome').get_text())
        score.append(match.find('th', class_='fscore').get_text())
        away.append(match.find('th', class_='faway').get_text())

    dict_football = {'Home': home, 'Score': score, 'Away': away}
    df_football = pd.DataFrame(dict_football)
    df_football['year'] = year
    return df_football

#Historical data
fifa = [get_matches(year) for year in years]
df_fifa = pd.concat(fifa, ignore_index=True)
df_fifa.to_csv('fifa_worldcupU20_historical_data.csv', index=False)

#Fixtures 2025
df_fixtures = get_matches(2025)
df_fixtures.to_csv('fifa_worldcupU20_fixtures_2025.csv', index=False)

In [14]:
df_fixtures = pd.read_csv('fifa_worldcupU20_fixtures_2025.csv')
df_historical_data = pd.read_csv('fifa_worldcupU20_historical_data.csv')

In [15]:
df_fixtures['Home'] = df_fixtures['Home'].str.strip()
df_fixtures['Away'] = df_fixtures['Away'].str.strip()    

In [16]:
import pandas as pd

# Load the CSV file
df_historical_data = pd.read_csv('fifa_worldcupU20_historical_data.csv')

# Filter rows where Score contains (a.e.t.)
df_historical_data['Score'] = df_historical_data['Score'].str.replace(r'\s*\(a\.e\.t\.\)', '', regex=True)

In [17]:
df_historical_data['Home'] = df_historical_data['Home'].str.strip()
df_historical_data['Away'] = df_historical_data['Away'].str.strip()

In [18]:
df_historical_data

,Home,Score,Away,year
0,France,1–2,Spain,1977
1,Mexico,6–0,Tunisia,1977
2,Spain,1–1,Mexico,1977
3,Tunisia,0–1,France,1977
4,France,1–1,Mexico,1977
...,...,...,...,...
987,United States,0–2,Uruguay,2023
988,Uruguay,1–0,Israel,2023
989,Italy,2–1,South Korea,2023
990,Israel,3–1,South Korea,2023


In [19]:
# Split the Score column using the en dash and assign to new columns
df_historical_data[['HomeGoals', 'AwayGoals']] = df_historical_data['Score'].str.split('–', expand=True)

# Convert to integer type
df_historical_data['HomeGoals'] = df_historical_data['HomeGoals'].astype(int)
df_historical_data['AwayGoals'] = df_historical_data['AwayGoals'].astype(int)

In [20]:
df_historical_data

,Home,Score,Away,year,HomeGoals,AwayGoals
0,France,1–2,Spain,1977,1,2
1,Mexico,6–0,Tunisia,1977,6,0
2,Spain,1–1,Mexico,1977,1,1
3,Tunisia,0–1,France,1977,0,1
4,France,1–1,Mexico,1977,1,1
...,...,...,...,...,...,...
987,United States,0–2,Uruguay,2023,0,2
988,Uruguay,1–0,Israel,2023,1,0
989,Italy,2–1,South Korea,2023,2,1
990,Israel,3–1,South Korea,2023,3,1


In [21]:
df_historical_data.drop('Score', axis=1, inplace=True)

In [22]:
df_historical_data.dtypes

Home         object
Away         object
year          int64
HomeGoals     int64
AwayGoals     int64
dtype: object

In [23]:
df_historical_data.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam', 'year': 'Year'}, inplace=True)


In [24]:
df_historical_data['TotalGoals'] = df_historical_data['HomeGoals'] + df_historical_data['AwayGoals']

In [25]:
df_historical_data.to_csv('clean_fifa_worldcupU20_historical_data.csv', index=False)
df_fixtures.to_csv('clean_fifa_worldcupU20_fixtures_2025.csv', index=False)

In [26]:
years = [1977, 1979, 1981, 1983, 1985, 1987, 1989, 1991, 1993,
        1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 
        2013, 2015, 2017, 2019, 2023] 

for year in years:
    print(year, len(df_historical_data[df_historical_data['Year'] == year]))

1977 28
1979 32
1981 32
1983 32
1985 32
1987 32
1989 32
1991 32
1993 32
1995 32
1997 52
1999 52
2001 52
2003 52
2005 52
2007 52
2009 52
2011 52
2013 52
2015 52
2017 52
2019 52
2023 52


In [27]:
import pandas as pd 
import pickle
from scipy.stats import poisson 

In [28]:
dict_table = pickle.load(open('dict_table', 'rb'))
df_historical_data = pd.read_csv('clean_fifa_worldcupU20_historical_data.csv')
df_fixtures = pd.read_csv('clean_fifa_worldcupU20_fixtures_2025.csv')

In [29]:
#split df into df_home and df_away
df_home = df_historical_data[['HomeTeam', 'HomeGoals', 'AwayGoals']]
df_away = df_historical_data[['AwayTeam', 'HomeGoals', 'AwayGoals']]

In [30]:
#rename columns
df_home = df_home.rename(columns={'HomeTeam': 'Team', 'HomeGoals': 'GoalsScored', 'AwayGoals': 'GoalsConceded'})
df_away = df_away.rename(columns={'AwayTeam': 'Team', 'HomeGoals': 'GoalsConceded', 'AwayGoals': 'GoalsScored'})

In [31]:
#concatenate df_home and df_away, group by Team and calculate mean
df_team_strength =pd.concat([df_home, df_away], ignore_index=True).groupby('Team').mean()
df_team_strength

,GoalsScored,GoalsConceded
Team,,
Algeria,0.500000,1.500000
Angola,0.750000,1.000000
Argentina,2.011628,0.825581
Australia,1.125000,1.589286
Austria,0.500000,1.650000
...,...,...
Venezuela,2.181818,0.727273
Vietnam,0.000000,2.000000
West Germany,2.166667,0.583333


In [85]:
def predict_points(home, away):
    if home in df_team_strength.index and away in df_team_strength.index:
        # goals scored * goals conceded
        lamb_home = df_team_strength.at[home,'GoalsScored'] * df_team_strength.at[away,'GoalsConceded']
        lamb_away = df_team_strength.at[away,'GoalsScored'] * df_team_strength.at[home,'GoalsConceded']
        prob_home, prob_away, prob_draw = 0,0,0
        for x in range(0, 11):
            for y in range(0, 11):
                p = poisson.pmf(x, lamb_home) * poisson.pmf(y, lamb_away)
                if x == y:
                    prob_draw += p
                elif x > y:
                    prob_home += p
                else:
                    prob_away += p
        
        points_home = 3 * prob_home +  prob_draw 
        points_away = 3 * prob_away +  prob_draw
        return float(points_home), float(points_away)
    else:
        return (0, 0)           

Testing Function

In [86]:
predict_points('Brazil', 'Mexico')
predict_points('Japan', 'Egypt')
predict_points('Colombia', 'Norway')

(1.558626412373716, 1.2559496950886695)

Predicting World Cup

In [4]:
#Splitting fixtures into group, knockout, quaterfinal......
df_fixtures_group_36 = df_fixtures[:36].copy()
df_fixtures_knockout = df_fixtures[36:44].copy()
df_fixtures_quaterfinal = df_fixtures[44:49].copy()
df_fixtures_semifinal = df_fixtures[49:50].copy()
df_fixtures_final = df_fixtures[:51].copy()

NameError: name 'df_fixtures' is not defined

In [144]:
df_fixtures_final

,Home,Score,Away,year
0,Japan,v,Egypt,2025
1,Chile,v,New Zealand,2025
2,Egypt,v,New Zealand,2025
3,Chile,v,Japan,2025
4,Egypt,v,Chile,2025
5,New Zealand,v,Japan,2025
6,South Korea,v,Ukraine,2025
7,Paraguay,v,Panama,2025
8,Panama,v,Ukraine,2025
9,South Korea,v,Paraguay,2025


In [113]:
for group in dict_table:
    print(dict_table[group]['Team'].values)

['Egypt' 'Japan' 'New Zealand' 'Chile (H)']
['Ukraine' 'Paraguay' 'South Korea' 'Panama']
['Brazil' 'Spain' 'Mexico' 'Morocco']
['Argentina' 'Italy' 'Australia' 'Cuba']
['France' 'United States' 'South Africa' 'New Caledonia']
['Colombia' 'Norway' 'Nigeria' 'Saudi Arabia']


In [114]:
def predict_goals(home, away):
    # Use average goals scored/conceded for prediction
    if home in df_team_strength.index and away in df_team_strength.index:
        goals_home = df_team_strength.at[home, 'GoalsScored'] * df_team_strength.at[away, 'GoalsConceded']
        goals_away = df_team_strength.at[away, 'GoalsScored'] * df_team_strength.at[home, 'GoalsConceded']
        return float(goals_home), float(goals_away)
    else:
        return (0, 0)

for group in dict_table.keys():
    for col in ['Pts', 'GoalsScored', 'GoalsConceded']:
        if col not in dict_table[group].columns:
            dict_table[group][col] = 0

    teams_in_group = dict_table[group]['Team'].values
    df_fixtures_group_6 = df_fixtures_group_36[df_fixtures_group_36['Home'].isin(teams_in_group)]
    for index, row in df_fixtures_group_6.iterrows(): 
        home, away = row['Home'], row['Away']
        points_home, points_away = predict_points(home, away)
        goals_home, goals_away = predict_goals(home, away)
        dict_table[group].loc[dict_table[group]['Team'] == home, 'Pts'] += points_home
        dict_table[group].loc[dict_table[group]['Team'] == away, 'Pts'] += points_away
        dict_table[group].loc[dict_table[group]['Team'] == home, 'GoalsScored'] += goals_home
        dict_table[group].loc[dict_table[group]['Team'] == home, 'GoalsConceded'] += goals_away
        dict_table[group].loc[dict_table[group]['Team'] == away, 'GoalsScored'] += goals_away
        dict_table[group].loc[dict_table[group]['Team'] == away, 'GoalsConceded'] += goals_home

    dict_table[group]['GD'] = dict_table[group]['GoalsScored'] - dict_table[group]['GoalsConceded']
    # Sort by Pts, then GD, then GoalsScored
    dict_table[group] = dict_table[group].sort_values(['Pts', 'GD', 'GoalsScored'], ascending=False).reset_index(drop=True)
    dict_table[group] = dict_table[group][['Team', 'Pts', 'GoalsScored', 'GoalsConceded', 'GD']]
    dict_table[group] = dict_table[group].round(0)

# Select winners and runners-up for knockout stage
for group in dict_table:
    group_winner = dict_table[group].iloc[0]['Team']
    runner_up = dict_table[group].iloc[1]['Team']
    df_fixtures_knockout.replace({f'Winners {group}': group_winner,
                                  f'Runners-up {group}': runner_up }, inplace=True)

In [115]:
# Collect all third-placed teams
third_placed = []
for group in dict_table:
    # Ensure sorting
    dict_table[group] = dict_table[group].sort_values(['Pts', 'GD', 'GoalsScored'], ascending=False).reset_index(drop=True)
    third_placed.append(dict_table[group].iloc[2])

# Create DataFrame and sort to get best third-placed teams
df_third = pd.DataFrame(third_placed)
df_third = df_third.sort_values(['Pts', 'GD', 'GoalsScored'], ascending=False).reset_index(drop=True)

# Select top N third-placed teams (N depends on your tournament format, e.g., 4 for 24 teams)
best_third_teams = df_third['Team'].tolist()[:4]

# Replace placeholders in knockout fixtures
for group in dict_table:
    group_winner = dict_table[group].iloc[0]['Team']
    runner_up = dict_table[group].iloc[1]['Team']
    df_fixtures_knockout.replace({f'Winners {group}': group_winner,
                                  f'Runners-up {group}': runner_up}, inplace=True)

# Replace third-placed placeholders (example for 4 best third-placed teams)
third_place_map = {
    '3rd Group A/C/D': best_third_teams[0],
    '3rd Group B/E/F': best_third_teams[1],
    '3rd Group C/D/E': best_third_teams[2],
    '3rd Group A/B/F': best_third_teams[3]
}
df_fixtures_knockout.replace(third_place_map, inplace=True)

df_fixtures_knockout

,Home,Score,Away,year
36,Japan,Match 37,Spain,2025
37,Ukraine,Match 39,Nigeria,2025
38,Argentina,Match 38,Australia,2025
39,Colombia,Match 40,United States,2025
40,Egypt,Match 44,South Korea,2025
41,Paraguay,Match 43,Nigeria,2025
42,France,Match 41,Italy,2025
43,Brazil,Match 42,Mexico,2025


In [116]:
dict_table['Group F']

,Team,Pts,GoalsScored,GoalsConceded,GD
0,Colombia,15.0,21.0,15.0,5.0
1,Norway,15.0,21.0,18.0,3.0
2,Nigeria,15.0,18.0,15.0,3.0
3,Saudi Arabia,6.0,12.0,21.0,-9.0


Knockout

In [117]:
df_fixtures_knockout

,Home,Score,Away,year
36,Japan,Match 37,Spain,2025
37,Ukraine,Match 39,Nigeria,2025
38,Argentina,Match 38,Australia,2025
39,Colombia,Match 40,United States,2025
40,Egypt,Match 44,South Korea,2025
41,Paraguay,Match 43,Nigeria,2025
42,France,Match 41,Italy,2025
43,Brazil,Match 42,Mexico,2025


In [119]:
# Upate the knockout with group winners and runners up
for group in dict_table:
    group_winner = dict_table[group].loc[0, 'Team']
    runner_up = dict_table[group].loc[1, 'Team']

    df_fixtures_knockout.replace({f'Winners {group}': group_winner,
                                  f'Runners-up {group}': runner_up }, inplace=True)

df_fixtures_knockout['winner'] = '?'
df_fixtures_knockout

,Home,Score,Away,year,winner
36,Japan,Match 37,Spain,2025,?
37,Ukraine,Match 39,Nigeria,2025,?
38,Argentina,Match 38,Australia,2025,?
39,Colombia,Match 40,United States,2025,?
40,Egypt,Match 44,South Korea,2025,?
41,Paraguay,Match 43,Nigeria,2025,?
42,France,Match 41,Italy,2025,?
43,Brazil,Match 42,Mexico,2025,?


In [120]:
#create get_winner Function
def get_winner(df_fixture_updated):
    for index, row in df_fixture_updated.iterrows():
        home, away = row['Home'], row['Away']
        points_home, points_away = predict_points(home, away)
        if points_home > points_away:
            winnner = home
        else:
            winnner = away
        df_fixture_updated.loc[index, 'winner'] = winnner
    return df_fixture_updated

In [121]:
get_winner(df_fixtures_knockout)

,Home,Score,Away,year,winner
36,Japan,Match 37,Spain,2025,Spain
37,Ukraine,Match 39,Nigeria,2025,Ukraine
38,Argentina,Match 38,Australia,2025,Argentina
39,Colombia,Match 40,United States,2025,Colombia
40,Egypt,Match 44,South Korea,2025,Egypt
41,Paraguay,Match 43,Nigeria,2025,Nigeria
42,France,Match 41,Italy,2025,France
43,Brazil,Match 42,Mexico,2025,Brazil


QuaterFInals


In [122]:
def update_table(df_fixtures_round_1, df_fixtures_round_2):
    for index, row in df_fixtures_round_1.iterrows():
        winner = df_fixtures_round_1.loc[index, 'winner']
        match = df_fixtures_round_1.loc[index, 'Score']
        df_fixtures_round_2.replace({f'Winners {match}': winner}, inplace=True)
    df_fixtures_round_2['winner'] = '?'
    return df_fixtures_round_2

In [123]:
update_table(df_fixtures_knockout, df_fixtures_quaterfinal)

,Home,Score,Away,year,winner
44,Ukraine,Match 46,Colombia,2025,?
45,Spain,Match 45,Argentina,2025,?
46,France,Match 47,Brazil,2025,?
47,Nigeria,Match 48,Egypt,2025,?


In [124]:
get_winner(df_fixtures_quaterfinal)

,Home,Score,Away,year,winner
44,Ukraine,Match 46,Colombia,2025,Ukraine
45,Spain,Match 45,Argentina,2025,Argentina
46,France,Match 47,Brazil,2025,Brazil
47,Nigeria,Match 48,Egypt,2025,Nigeria


In [125]:
update_table(df_fixtures_quaterfinal, df_fixtures_semifinal)

,Home,Score,Away,year,winner
48,Brazil,Match 50,Nigeria,2025,?
49,Argentina,Match 49,Ukraine,2025,?


In [126]:
get_winner(df_fixtures_semifinal)

,Home,Score,Away,year,winner
48,Brazil,Match 50,Nigeria,2025,Brazil
49,Argentina,Match 49,Ukraine,2025,Argentina


In [127]:
update_table(df_fixtures_semifinal, df_fixtures_final)

,Home,Score,Away,year,winner
0,Japan,v,Egypt,2025,?
1,Chile,v,New Zealand,2025,?
2,Egypt,v,New Zealand,2025,?
3,Chile,v,Japan,2025,?
4,Egypt,v,Chile,2025,?
5,New Zealand,v,Japan,2025,?
6,South Korea,v,Ukraine,2025,?
7,Paraguay,v,Panama,2025,?
8,Panama,v,Ukraine,2025,?
9,South Korea,v,Paraguay,2025,?


In [128]:
get_winner(df_fixtures_final)

,Home,Score,Away,year,winner
0,Japan,v,Egypt,2025,Egypt
1,Chile,v,New Zealand,2025,Chile
2,Egypt,v,New Zealand,2025,Egypt
3,Chile,v,Japan,2025,Japan
4,Egypt,v,Chile,2025,Egypt
5,New Zealand,v,Japan,2025,Japan
6,South Korea,v,Ukraine,2025,Ukraine
7,Paraguay,v,Panama,2025,Paraguay
8,Panama,v,Ukraine,2025,Ukraine
9,South Korea,v,Paraguay,2025,Paraguay
